# Electric Vehicle Population Analysis

**Category:** 02-Data-Exploration / CSV Datasources  
**Dataset:** Electric_Vehicle_Population_Data.csv (57MB)  
**Source:** Washington State DOL  

---

## Dataset Overview
- EV registration data with make, model, year, location
- Geographic coordinates for mapping
- Ideal for: Market analysis, adoption trends, geospatial visualization

In [ ]:
%pip install -q pandas numpy matplotlib seaborn folium

CLAUDE = "anthropic-chat:claude-sonnet-4-5-20250929"
MODEL = CLAUDE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
DATA_PATH = Path.home() / 'lab' / 'data' / 'CSVs' / 'Electric_Vehicle_Population_Data.csv'

## 1. Load and Explore

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} EV registrations")
df.head()

In [ ]:
df.info()

In [ ]:
# Key statistics
print(f"Model Years: {df['Model Year'].min()} - {df['Model Year'].max()}")
print(f"Unique Makes: {df['Make'].nunique()}")
print(f"Unique Models: {df['Model'].nunique()}")
print(f"Counties: {df['County'].nunique()}")

## 2. EV Type Distribution

In [ ]:
# BEV vs PHEV
ev_types = df['Electric Vehicle Type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(ev_types.values, labels=ev_types.index, autopct='%1.1f%%', 
            colors=['#2ecc71', '#3498db'])
axes[0].set_title('EV Type Distribution')

# CAFV eligibility
cafv = df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].value_counts()
axes[1].barh(cafv.index, cafv.values, color='teal')
axes[1].set_title('CAFV Eligibility')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 3. Market Share Analysis

In [ ]:
# Top manufacturers
top_makes = df['Make'].value_counts().head(10)

plt.figure(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0, 0.8, len(top_makes)))
top_makes.plot(kind='bar', color=colors)
plt.title('Top 10 EV Manufacturers')
plt.xlabel('Make')
plt.ylabel('Number of Registrations')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\nTesla market share: {top_makes['TESLA'] / len(df) * 100:.1f}%")

In [ ]:
# Top models
df['Make_Model'] = df['Make'] + ' ' + df['Model']
top_models = df['Make_Model'].value_counts().head(15)

plt.figure(figsize=(12, 8))
top_models.plot(kind='barh', color='steelblue')
plt.title('Top 15 EV Models')
plt.xlabel('Registrations')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Adoption Trends Over Time

In [ ]:
# Registrations by model year
yearly = df['Model Year'].value_counts().sort_index()
yearly = yearly[yearly.index >= 2010]  # Focus on modern EVs

plt.figure(figsize=(12, 6))
plt.bar(yearly.index, yearly.values, color='#27ae60', edgecolor='white')
plt.title('EV Registrations by Model Year')
plt.xlabel('Model Year')
plt.ylabel('Number of Vehicles')

# Add growth annotation
for i, (year, count) in enumerate(yearly.items()):
    if year >= 2018:
        plt.annotate(f'{count:,}', (year, count), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# BEV vs PHEV trend over time
type_by_year = df.groupby(['Model Year', 'Electric Vehicle Type']).size().unstack(fill_value=0)
type_by_year = type_by_year[type_by_year.index >= 2010]

type_by_year.plot(kind='bar', stacked=True, figsize=(14, 6), 
                   color=['#2ecc71', '#3498db'])
plt.title('BEV vs PHEV Adoption by Year')
plt.xlabel('Model Year')
plt.ylabel('Registrations')
plt.legend(title='Type')
plt.tight_layout()
plt.show()

In [ ]:
%%ai $MODEL
Analyzing EV adoption trends:

1. What factors drove the explosive growth in EV registrations after 2018?
2. Why does Tesla dominate the market? What could competitors do?
3. What's the significance of BEV vs PHEV trends?
4. Predict what EV adoption might look like in 2030.

## 5. Geographic Analysis

In [ ]:
# Top counties
county_counts = df['County'].value_counts().head(15)

plt.figure(figsize=(12, 6))
county_counts.plot(kind='bar', color='coral')
plt.title('EV Registrations by County')
plt.xlabel('County')
plt.ylabel('Registrations')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Top cities
city_counts = df['City'].value_counts().head(20)

plt.figure(figsize=(12, 8))
city_counts.plot(kind='barh', color='purple')
plt.title('Top 20 Cities for EV Ownership')
plt.xlabel('Registrations')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Electric Range Analysis

In [ ]:
# Electric range distribution
range_data = df['Electric Range'].dropna()
range_data = range_data[range_data > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(range_data, bins=50, color='green', edgecolor='white', alpha=0.7)
axes[0].axvline(range_data.median(), color='red', linestyle='--', label=f'Median: {range_data.median():.0f} mi')
axes[0].set_title('Electric Range Distribution')
axes[0].set_xlabel('Range (miles)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Range by year
range_by_year = df[df['Electric Range'] > 0].groupby('Model Year')['Electric Range'].median()
range_by_year = range_by_year[range_by_year.index >= 2012]

axes[1].plot(range_by_year.index, range_by_year.values, 'bo-', linewidth=2, markersize=8)
axes[1].set_title('Median Electric Range by Model Year')
axes[1].set_xlabel('Model Year')
axes[1].set_ylabel('Range (miles)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Base MSRP Analysis

In [ ]:
# Price distribution
price_data = df['Base MSRP'].dropna()
price_data = price_data[(price_data > 0) & (price_data < 200000)]  # Clean outliers

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(price_data, bins=50, color='gold', edgecolor='white')
axes[0].axvline(price_data.median(), color='red', linestyle='--', label=f'Median: ${price_data.median():,.0f}')
axes[0].set_title('Base MSRP Distribution')
axes[0].set_xlabel('Price ($)')
axes[0].legend()

# Price vs Range
sample = df[(df['Base MSRP'] > 0) & (df['Electric Range'] > 0)].sample(n=min(5000, len(df)))
axes[1].scatter(sample['Base MSRP'], sample['Electric Range'], alpha=0.3, s=10)
axes[1].set_title('Price vs Electric Range')
axes[1].set_xlabel('Base MSRP ($)')
axes[1].set_ylabel('Electric Range (miles)')

plt.tight_layout()
plt.show()

---

## Summary

This notebook analyzed:
- EV type distribution (BEV vs PHEV)
- Manufacturer and model market share
- Adoption trends over time
- Geographic distribution
- Electric range improvements
- Price analysis

**Key Insights:**
- Tesla dominates with ~60%+ market share
- BEV adoption is growing faster than PHEV
- Electric range has improved significantly (median ~100+ miles)
- Urban areas lead EV adoption

---